# **Manufacturing Equipment Output Prediction**
Linear Regression capstone project to predict hourly output (Parts_Per_Hour) from injection molding machine parameters.



# 1. Import **libraries**

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2.Load **Dataset**

In [16]:
from google.colab import files
import io

uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(df.shape)
df.head()

Saving manufacturing_dataset_1000_samples(2).csv to manufacturing_dataset_1000_samples(2) (1).csv
Uploaded file: manufacturing_dataset_1000_samples(2) (1).csv
(1000, 19)


,Timestamp,Injection_Temperature,Injection_Pressure,Cycle_Time,Cooling_Time,Material_Viscosity,Ambient_Temperature,Machine_Age,Operator_Experience,Maintenance_Hours,Shift,Machine_Type,Material_Grade,Day_of_Week,Temperature_Pressure_Ratio,Total_Cycle_Time,Efficiency_Score,Machine_Utilization,Parts_Per_Hour
0,2023-01-01 00:00:00,221.0,136.0,28.7,13.6,375.5,28.0,3.8,11.2,64,Evening,Type_B,Economy,Thursday,1.625,42.3,0.063,0.510,36.5
1,2023-01-01 01:00:00,213.3,128.9,34.5,14.0,215.8,22.6,6.8,6.3,58,Night,Type_A,Standard,Wednesday,1.655,48.5,0.037,0.389,29.9
2,2023-01-01 02:00:00,222.8,115.9,19.9,9.5,307.0,25.3,4.2,9.6,47,Day,Type_A,Standard,Monday,1.922,29.4,0.061,0.551,56.9
3,2023-01-01 03:00:00,233.3,105.3,39.2,13.1,137.8,26.0,9.2,8.6,49,Evening,Type_A,Premium,Saturday,2.215,52.3,0.054,0.293,31.0
4,2023-01-01 04:00:00,212.2,125.5,45.0,9.9,298.2,23.6,6.2,23.0,49,Night,Type_B,Premium,Monday,1.691,54.9,0.145,0.443,15.0


# **3.Basic Data Understanding**

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Timestamp                   1000 non-null   object 
 1   Injection_Temperature       1000 non-null   float64
 2   Injection_Pressure          1000 non-null   float64
 3   Cycle_Time                  1000 non-null   float64
 4   Cooling_Time                1000 non-null   float64
 5   Material_Viscosity          980 non-null    float64
 6   Ambient_Temperature         980 non-null    float64
 7   Machine_Age                 1000 non-null   float64
 8   Operator_Experience         980 non-null    float64
 9   Maintenance_Hours           1000 non-null   int64  
 10  Shift                       1000 non-null   object 
 11  Machine_Type                1000 non-null   object 
 12  Material_Grade              1000 non-null   object 
 13  Day_of_Week                 1000 n

# 4.Check Missing **Values**

In [18]:
df.isnull().sum()

,0
Timestamp,0
Injection_Temperature,0
Injection_Pressure,0
Cycle_Time,0
Cooling_Time,0
Material_Viscosity,20
Ambient_Temperature,20
Machine_Age,0
Operator_Experience,20
Maintenance_Hours,0


# 5.Define Features and Target **Variable**

In [19]:
X = df.drop("Parts_Per_Hour", axis=1)
y = df["Parts_Per_Hour"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 18)
y shape: (1000,)


# **6.Timestamp Feature Engineering**

In [20]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df["Hour"] = df["Timestamp"].dt.hour
df["Month"] = df["Timestamp"].dt.month
df["Day"] = df["Timestamp"].dt.day

df = df.drop("Timestamp", axis=1)

df.head()

,Injection_Temperature,Injection_Pressure,Cycle_Time,Cooling_Time,Material_Viscosity,Ambient_Temperature,Machine_Age,Operator_Experience,Maintenance_Hours,Shift,...,Material_Grade,Day_of_Week,Temperature_Pressure_Ratio,Total_Cycle_Time,Efficiency_Score,Machine_Utilization,Parts_Per_Hour,Hour,Month,Day
0,221.0,136.0,28.7,13.6,375.5,28.0,3.8,11.2,64,Evening,...,Economy,Thursday,1.625,42.3,0.063,0.510,36.5,0,1,1
1,213.3,128.9,34.5,14.0,215.8,22.6,6.8,6.3,58,Night,...,Standard,Wednesday,1.655,48.5,0.037,0.389,29.9,1,1,1
2,222.8,115.9,19.9,9.5,307.0,25.3,4.2,9.6,47,Day,...,Standard,Monday,1.922,29.4,0.061,0.551,56.9,2,1,1
3,233.3,105.3,39.2,13.1,137.8,26.0,9.2,8.6,49,Evening,...,Premium,Saturday,2.215,52.3,0.054,0.293,31.0,3,1,1
4,212.2,125.5,45.0,9.9,298.2,23.6,6.2,23.0,49,Night,...,Premium,Monday,1.691,54.9,0.145,0.443,15.0,4,1,1


# 7.Identify Numerical and Categorical **Features**

In [21]:
X = df.drop("Parts_Per_Hour", axis=1)
y = df["Parts_Per_Hour"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (1000, 20)
Target shape: (1000,)


# 8.Data **Preprocessing**

In [22]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numerical columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numerical columns:
Index(['Injection_Temperature', 'Injection_Pressure', 'Cycle_Time',
       'Cooling_Time', 'Material_Viscosity', 'Ambient_Temperature',
       'Machine_Age', 'Operator_Experience', 'Maintenance_Hours',
       'Temperature_Pressure_Ratio', 'Total_Cycle_Time', 'Efficiency_Score',
       'Machine_Utilization'],
      dtype='object')

Categorical columns:
Index(['Shift', 'Machine_Type', 'Material_Grade', 'Day_of_Week'], dtype='object')


# 9.Train-Test **Split**

In [23]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 10.Build and Train Linear Regression **Model**

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (800, 20)
Testing data: (200, 20)


In [25]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model training completed successfully")

Model training completed successfully


# 12.**Make** Predictions

In [26]:
y_pred = model.predict(X_test)

print("Predictions completed")
print(y_pred[:10])

Predictions completed
[19.90192601 44.40634188 34.04944202 39.68841147 18.39527151 20.43364841
 33.5421094  14.1105141  27.97829494 32.34135426]


# 13.Model **Evaluation**

In [27]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE : 2.7171638441002917
MSE : 12.304517848532386
RMSE: 3.507779618010856
R2 Score: 0.9057073405009888
